# Sisyphus Humanoid — Training Notebook

Train a humanoid to push a rock up an incline using PPO.

**Architecture:** MuJoCo → PPO (Stable-Baselines3) → Trajectory Logger → Preview Renderer

**Curriculum:** Phase I (flat, light) → Phase II (5°, medium) → Phase III (10°, heavy) → Phase IV (15°, infinite loop)

## 1. Setup

In [ ]:
# Install dependencies (torch with CUDA for A100 GPU acceleration)
!pip install -q mujoco gymnasium stable-baselines3 imageio[ffmpeg] tensorboard torch

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone or update the repo
import os

REPO_URL = 'https://github.com/Mrepp/Sisyphus.git'
REPO_DIR = '/content/Sisyphus'
DRIVE_DIR = '/content/drive/MyDrive/sisyphus'

# Create Drive directory for persistent data
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/replays', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/renders_preview', exist_ok=True)

# Clone repo (pulls latest from GitHub)
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

# Add to Python path
import sys
sys.path.insert(0, REPO_DIR)

In [ ]:
# Verify MuJoCo works
import mujoco
print(f'MuJoCo version: {mujoco.__version__}')

# Quick env test
from env.sisyphus_env import SisyphusEnv

test_env = SisyphusEnv(slope_deg=0.0, rock_mass=5.0)
obs, info = test_env.reset()
print(f'Observation shape: {obs.shape}')
print(f'Action space: {test_env.action_space}')

# Quick random step test
for i in range(10):
    action = test_env.action_space.sample()
    obs, reward, terminated, truncated, info = test_env.step(action)
print(f'10 random steps OK. Last reward: {reward:.4f}')
test_env.close()

## 2. Configuration

In [ ]:
from train.train_ppo import get_hardware_config

# Auto-detect hardware and get recommended settings
hw_config = get_hardware_config(profile="auto")
# Override if needed: hw_config = get_hardware_config(profile="colab_a100", num_envs=10)

NUM_ENVS = hw_config["num_envs"]
TOTAL_TIMESTEPS = 50_000_000  # Total training steps (adjust for testing)
CHECKPOINT_FREQ = 500_000  # Save every N steps

# Directories (on Google Drive for persistence)
CHECKPOINT_DIR = f'{DRIVE_DIR}/checkpoints'
REPLAY_DIR = f'{DRIVE_DIR}/replays'
RENDER_DIR = f'{DRIVE_DIR}/renders_preview'
TB_LOG_DIR = f'{DRIVE_DIR}/tensorboard'

print(f'Hardware profile: {hw_config["profile"]}')
print(f'Device: {hw_config["device"]}')
print(f'Network: {hw_config["net_arch"]}')
print(f'Training {TOTAL_TIMESTEPS:,} steps with {NUM_ENVS} envs')
print(f'n_steps: {hw_config["n_steps"]}, batch_size: {hw_config["batch_size"]}')
print(f'Rollout buffer: {NUM_ENVS * hw_config["n_steps"]:,} transitions')

## 3. Create Environment & Model

In [ ]:
from train.train_ppo import create_env, create_model, setup_torch_optimizations

# IMPORTANT: Create vectorized environment BEFORE initializing torch/CUDA.
# SubprocVecEnv with start_method="spawn" requires that CUDA is NOT yet
# initialized in the parent process, otherwise children inherit stale state.
vec_env = create_env(num_envs=NUM_ENVS, slope=0.0, rock_mass=5.0)

# Now safe to initialize CUDA optimizations in the parent process
setup_torch_optimizations()

# Create PPO model with hardware-optimized settings
model = create_model(vec_env, tensorboard_log=TB_LOG_DIR, hardware_config=hw_config)

print(f'Model device: {model.device}')
print(f'Policy arch: {model.policy.net_arch}')
print(f'Rollout buffer: {model.n_steps * NUM_ENVS:,} transitions')
print(f'Minibatches per epoch: {model.n_steps * NUM_ENVS // model.batch_size}')

In [ ]:
# Optional: Resume from checkpoint
# from stable_baselines3 import PPO
# model = PPO.load(f'{CHECKPOINT_DIR}/sisyphus_ppo_5000000_steps', env=vec_env)

## 4. Train

In [ ]:
import threading
import time
from train.train_ppo import train, save_vec_normalize
from train.curriculum import CurriculumManager

# Colab keepalive: prevent WebSocket timeout during long training runs
_keepalive_active = True

def _colab_keepalive(interval=60):
    while _keepalive_active:
        time.sleep(interval)
        if _keepalive_active:
            print(".", end="", flush=True)

keepalive_thread = threading.Thread(target=_colab_keepalive, daemon=True)
keepalive_thread.start()

# Create evaluation env (for trajectory logging callbacks)
eval_env = SisyphusEnv(slope_deg=0.0, rock_mass=5.0, max_steps=1000)
curriculum = CurriculumManager()

# Train in chunks to avoid single long blocking call.
# Each chunk saves a checkpoint, allowing recovery if the runtime disconnects.
CHUNK_SIZE = 2_000_000
steps_done = 0
first_chunk = True

while steps_done < TOTAL_TIMESTEPS:
    chunk_steps = min(CHUNK_SIZE, TOTAL_TIMESTEPS - steps_done)
    print(f"\n--- Training chunk: {steps_done:,} to {steps_done + chunk_steps:,} ---")

    model = train(
        model,
        total_timesteps=chunk_steps,
        checkpoint_dir=CHECKPOINT_DIR,
        replay_dir=REPLAY_DIR,
        render_dir=RENDER_DIR,
        checkpoint_freq=CHECKPOINT_FREQ,
        use_curriculum=True,
        curriculum=curriculum,
        eval_env=eval_env,
        render_enabled=False,
        reset_num_timesteps=first_chunk,
    )

    steps_done += chunk_steps
    first_chunk = False

    # Save intermediate checkpoint to Drive (survives runtime restart)
    model.save(f'{CHECKPOINT_DIR}/sisyphus_ppo_step_{steps_done}')
    save_vec_normalize(vec_env, f'{CHECKPOINT_DIR}/vec_normalize_step_{steps_done}.pkl')
    print(f"Checkpoint saved at step {steps_done:,}")

_keepalive_active = False
print(f"\nTraining complete: {steps_done:,} total steps")

In [ ]:
# Save final model + normalization stats
from train.train_ppo import save_vec_normalize

model.save(f'{CHECKPOINT_DIR}/sisyphus_ppo_final')
save_vec_normalize(vec_env, f'{CHECKPOINT_DIR}/vec_normalize_final.pkl')
print('Final model saved.')

## 5. Monitor Training

In [ ]:
# Launch TensorBoard inline
%load_ext tensorboard
%tensorboard --logdir {TB_LOG_DIR}

## 6. Quick Evaluation

In [ ]:
# Run a quick evaluation and display preview
from IPython.display import Video
from render.preview_renderer import PreviewRenderer

eval_env = SisyphusEnv(slope_deg=10.0, rock_mass=10.0, max_steps=500)
renderer = PreviewRenderer(eval_env.model, width=960, height=540)

mp4_path = '/content/quick_eval.mp4'
renderer.render_episode(eval_env, model, mp4_path, fps=30, max_steps=500)
renderer.close()
eval_env.close()

Video(mp4_path, embed=True, width=960)

In [ ]:
# List saved files on Drive
import glob

print('Checkpoints:')
for f in sorted(glob.glob(f'{CHECKPOINT_DIR}/*.zip')):
    print(f'  {os.path.basename(f)}')

print(f'\nTrajectories: {len(glob.glob(f"{REPLAY_DIR}/*.npz"))} files')
print(f'Previews: {len(glob.glob(f"{RENDER_DIR}/*.mp4"))} files')